# Module 4 — Multi-Agent Orchestration

**Goal:** coordinate five specialised agents (Orchestrator, Retriever, Reasoner, Symbolic Verifier, Auditor) via LangGraph, producing a clinical recommendation with each claim traceable to the knowledge graph.

**Strategy:** prove the full pipeline on `deepseek-r1:7b` via Ollama first. `deepseek-r1:32b` needs roughly 22–24GB VRAM, which does not fit on Colab's free T4 (16GB VRAM) — the same limit already documented in `00_setup.ipynb`. A 32b run would require Colab Pro's A100 (40GB VRAM). Since this is a paid, occasional resource rather than something to depend on for routine runs, the pipeline is proven end-to-end on 7b first; swapping in 32b later, on an A100 session, is then a small, isolated change to the Reasoner agent's model call, not something that reshapes the whole orchestration.

This is the Colab port of the local version. Fresh Colab runtimes do not carry over Ollama or any pip packages from other notebooks, so this notebook installs and starts everything it needs on its own, the same way `00_setup.ipynb` does.

Requires Modules 1–3's outputs (FAISS index, chunk metadata, populated Neo4j graph) to already exist.

## 1. Mount Drive

In [1]:
from google.colab import drive
drive.mount('/content/drive')

PROJECT_ROOT = "/content/drive/MyDrive/ClinicalTrust"
PROCESSED_DIR = f"{PROJECT_ROOT}/data/processed"

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


## 2. Install FAISS on its own

Kept isolated from the other installs, same reasoning as `00_setup.ipynb`: bundling `faiss-gpu` (or anything that might fail to build) into a combined pip line can silently abort every package after it in that line.

In [2]:
!pip install -q faiss-cpu

## 3. Install and start Ollama, then pull `deepseek-r1:7b`

Same sequence as `00_setup.ipynb`: install the system dependency, install Ollama, start the server as a background process, and actively poll until it responds before pulling — rather than guessing with a fixed sleep.

In [3]:
!apt-get install -y zstd
!curl -fsSL https://ollama.com/install.sh | sh

Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
zstd is already the newest version (1.4.8+dfsg-3build1).
0 upgraded, 0 newly installed, 0 to remove and 57 not upgraded.
>>> Cleaning up old version at /usr/local/lib/ollama
>>> Installing ollama to /usr/local
>>> Downloading ollama-linux-amd64.tar.zst
######################################################################## 100.0%
>>> Adding ollama user to video group...
>>> Adding current user to ollama group...
>>> Creating ollama systemd service...
>>> The Ollama API is now available at 127.0.0.1:11434.
>>> Install complete. Run "ollama" from the command line.


In [4]:
import subprocess, time, requests

ollama_process = subprocess.Popen(["ollama", "serve"])

for attempt in range(30):
    try:
        requests.get("http://127.0.0.1:11434")
        print("Ollama server is up.")
        break
    except requests.exceptions.ConnectionError:
        time.sleep(2)
else:
    raise RuntimeError("Ollama server did not start in time — re-run this cell.")

Ollama server is up.


In [5]:
!ollama pull deepseek-r1:7b

## 4. Install the remaining Python packages

`nmslib` is required by scispaCy's UMLS EntityLinker and is not pulled in automatically by a plain `pip install scispacy`. The model install stays separate with `--no-deps`, for the same reason established in Module 2 — this old model release pins a spaCy version with no wheel for Colab's current Python.

In [6]:
!pip install -q transformers neo4j langgraph python-dotenv ollama
!pip install -q nmslib-metabrainz==2.1.3
!pip install -q --no-deps scispacy
!pip install -q conllu pysbd scikit-learn scipy joblib

In [7]:
!pip install -q --no-deps https://s3-us-west-2.amazonaws.com/ai2-s2-scispacy/releases/v0.5.4/en_core_sci_sm-0.5.4.tar.gz

  Preparing metadata (setup.py) ... done


## 5. Imports, model config patch, and Neo4j connection

In [8]:
import os
import gc
import glob
import re
import json
import pandas as pd
import numpy as np
import faiss
import torch
import en_core_sci_sm
from transformers import AutoTokenizer, AutoModel
from neo4j import GraphDatabase
import ollama
from typing import TypedDict, List
from langgraph.graph import StateGraph, END
from google.colab import userdata

# --- Patch the scispaCy model config before loading (same fix as Modules 2 & 3) ---
pkg_dir = os.path.dirname(en_core_sci_sm.__file__)
for path in glob.glob(os.path.join(pkg_dir, "**", "config.cfg"), recursive=True):
    with open(path, "r") as f:
        content = f.read()
    fixed = re.sub(r'=\s*"True"', "= true", content)
    fixed = re.sub(r'=\s*"False"', "= false", fixed)
    if fixed != content:
        with open(path, "w") as f:
            f.write(fixed)

# --- Neo4j Aura connection ---
NEO4J_URI = userdata.get('NEO4J_URI')
NEO4J_USERNAME = userdata.get('NEO4J_USERNAME')
NEO4J_PASSWORD = userdata.get('NEO4J_PASSWORD')
driver = GraphDatabase.driver(NEO4J_URI, auth=(NEO4J_USERNAME, NEO4J_PASSWORD))

MODEL_NAME = "emilyalsentzer/Bio_ClinicalBERT"
OLLAMA_MODEL = "deepseek-r1:7b"  # swap to a Colab Pro A100 32b run later, isolated to reasoning_agent

print("Setup ready.")

/usr/local/lib/python3.13/dist-packages/spacy/util.py:971: UserWarning: [W095] Model 'en_core_sci_sm' (0.5.4) was trained with spaCy v3.7.4 and may not be 100% compatible with the current version (3.8.15). If you see errors or degraded performance, download a newer compatible model or retrain your custom model with the current spaCy version. For more details and available updates, run: python -m spacy validate
  warnings.warn(warn_msg)


Setup ready.


## 6. Load retrieval components (Module 1 + Module 3 logic, condensed)

`embed_query` uses attention-masked mean pooling, matching the corrected version from Modules 1 and 3 — padding tokens are excluded from the average rather than included in it, so this stays consistent with however the corpus vectors in the FAISS index were actually computed.

In [9]:
faiss_index = faiss.read_index(os.path.join(PROCESSED_DIR, "pmc_patients.index"))
chunks_df = pd.read_parquet(os.path.join(PROCESSED_DIR, "chunks_metadata.parquet"))

from scispacy.linking import EntityLinker

device = "cuda" if torch.cuda.is_available() else "cpu"
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
bert_model = AutoModel.from_pretrained(MODEL_NAME).to(device)
bert_model.eval()

nlp = en_core_sci_sm.load()
nlp.add_pipe("scispacy_linker", config={"resolve_abbreviations": True, "linker_name": "umls"})
linker = nlp.get_pipe("scispacy_linker")

def embed_query(text):
    inputs = tokenizer([text], padding=True, truncation=True, max_length=512, return_tensors="pt").to(device)
    with torch.no_grad():
        outputs = bert_model(**inputs)

    mask = inputs["attention_mask"].unsqueeze(-1)
    summed = torch.sum(outputs.last_hidden_state * mask, dim=1)
    counts = torch.clamp(mask.sum(dim=1), min=1e-9)
    mean_pooled = summed / counts

    return mean_pooled.cpu().numpy()

def vector_search(query, k=10):
    query_vec = embed_query(query).astype("float32")
    distances, indices = faiss_index.search(query_vec, k)
    results = []
    for rank, idx in enumerate(indices[0]):
        row = chunks_df.iloc[idx]
        results.append({"patient_id": row["patient_id"], "rank": rank + 1, "text": row["text"]})
    df = pd.DataFrame(results).sort_values("rank").drop_duplicates(subset="patient_id", keep="first")
    return df.reset_index(drop=True)

def extract_query_cuis(query):
    doc = nlp(query)
    return list({ent._.kb_ents[0][0] for ent in doc.ents if ent._.kb_ents})

def graph_search(query, k=10):
    cuis = extract_query_cuis(query)
    if not cuis:
        return pd.DataFrame(columns=["patient_id", "matched_concepts", "rank"])
    with driver.session() as session:
        result = session.run(
            """
            MATCH (c:Concept)<-[:MENTIONS]-(p:Patient)
            WHERE c.cui IN $cuis
            RETURN p.patient_id AS patient_id, count(DISTINCT c) AS matched_concepts,
                   collect(DISTINCT c.canonical_name) AS concepts
            ORDER BY matched_concepts DESC LIMIT $k
            """,
            cuis=cuis, k=k
        )
        records = [dict(r) for r in result]
    df = pd.DataFrame(records)
    if not df.empty:
        df["rank"] = range(1, len(df) + 1)
    return df

print("Retrieval components loaded.")

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertModel LOAD REPORT from: emilyalsentzer/Bio_ClinicalBERT
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.predictions.decoder.weight             | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
/usr/local/lib/python3.13/dist-packages/spacy/util.py:971: UserWarning: [W095] Model 'en_core_sci_sm' (0.5.4) was trained with spaCy v3.7.4 and may not be 100% compatible with the curr

https://s3-us-west-2.amazonaws.com/ai2-s2-scispacy/data/linkers/2023-04-23/umls/tfidf_vectors_sparse.npz not found in cache, downloading to /tmp/tmpi7ve6w30


100%|██████████| 492M/492M [00:15<00:00, 34.1MiB/s]


Finished download, copying /tmp/tmpi7ve6w30 to cache at /root/.scispacy/datasets/2b79923846fb52e62d686f2db846392575c8eb5b732d9d26cd3ca9378c622d40.87bd52d0f0ee055c1e455ef54ba45149d188552f07991b765da256a1b512ca0b.tfidf_vectors_sparse.npz
https://s3-us-west-2.amazonaws.com/ai2-s2-scispacy/data/linkers/2023-04-23/umls/nmslib_index.bin not found in cache, downloading to /tmp/tmppuya7a4_


...

Finished download, copying /tmp/tmpc2dd_pzd to cache at /root/.scispacy/datasets/6238f505f56aca33290aab44097f67dd1b88880e3be6d6dcce65e56e9255b7d4.d7f77b1629001b40f1b1bc951f3a890ff2d516fb8fbae3111b236b31b33d6dcf.concept_aliases.json
https://s3-us-west-2.amazonaws.com/ai2-s2-scispacy/data/kbs/2023-04-23/umls_2022_ab_cat0129.jsonl not found in cache, downloading to /tmp/tmpr45j550h


100%|██████████| 628M/628M [01:18<00:00, 8.43MiB/s]


Finished download, copying /tmp/tmpr45j550h to cache at /root/.scispacy/datasets/d5e593bc2d8adeee7754be423cd64f5d331ebf26272074a2575616be55697632.0660f30a60ad00fffd8bbf084a18eb3f462fd192ac5563bf50940fc32a850a3c.umls_2022_ab_cat0129.jsonl
https://s3-us-west-2.amazonaws.com/ai2-s2-scispacy/data/umls_semantic_type_tree.tsv not found in cache, downloading to /tmp/tmptx3gmv18


100%|██████████| 4.26k/4.26k [00:00<00:00, 10.0MiB/s]

Finished download, copying /tmp/tmptx3gmv18 to cache at /root/.scispacy/datasets/21a1012c532c3a431d60895c509f5b4d45b0f8966c4178b892190a302b21836f.330707f4efe774134872b9f77f0e3208c1d30f50800b3b39a6b8ec21d9adf1b7.umls_semantic_type_tree.tsv
Retrieval components loaded.


## 7. Define the shared agent state

LangGraph passes a single state object between agents. Each agent reads what it needs and adds its own output.

In [10]:
class AgentState(TypedDict):
    query: str
    retrieved_context: str
    graph_concepts: List[str]
    raw_recommendation: str
    claims: List[dict]
    verified_claims: List[dict]
    audit_report: str

## 8. Orchestrator Agent

Receives the query, does light preprocessing (here: just passes it through — extend later with constraint extraction if needed).

In [11]:
def orchestrator_agent(state: AgentState) -> AgentState:
    print(f"[Orchestrator] Received query: {state['query']}")
    return state

## 9. Retriever Agent

Runs the dual-pathway retrieval from Module 3, merges results into a text context block for the Reasoner.

In [12]:
def retriever_agent(state: AgentState) -> AgentState:
    query = state["query"]
    vec_results = vector_search(query, k=5)
    graph_results = graph_search(query, k=5)

    context_parts = []
    for _, row in vec_results.iterrows():
        context_parts.append(f"Patient {row['patient_id']}: {row['text'][:400]}")

    concepts = []
    if not graph_results.empty:
        for _, row in graph_results.iterrows():
            concepts.extend(row["concepts"])

    print(f"[Retriever] {len(vec_results)} vector matches, {len(graph_results)} graph matches")

    state["retrieved_context"] = "\n\n".join(context_parts)
    state["graph_concepts"] = list(set(concepts))
    return state

## 10. Free retrieval models before loading the LLM

This mattered most on the local 8GB RAM machine; on Colab's T4 session RAM and VRAM are more comfortable, but running Bio-ClinicalBERT and a 7b Ollama model at once is still unnecessary overhead, so the same freeing step is kept rather than dropped.

In [13]:
def free_retrieval_memory():
    global bert_model
    del bert_model
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
    print("Freed Bio-ClinicalBERT from memory (scispaCy + FAISS index kept, lighter footprint).")

## 11. Reasoning Agent

Calls the local Ollama model, asking it to generate a recommendation grounded only in the retrieved context, tagging each claim. This prompt structure is what later points at a 32b run instead, once one is available.

In [14]:
REASONER_PROMPT_TEMPLATE = """You are a clinical reasoning assistant. Using ONLY the patient context below, \
answer the clinician's query. For each factual claim you make, tag it clearly like this: \
[CLAIM: your claim text]. If you cannot support a claim from the context, do not state it.

Patient context:
{context}

Known related concepts from the knowledge graph: {concepts}

Clinician query: {query}

Provide a short clinical recommendation with tagged claims."""

def reasoning_agent(state: AgentState) -> AgentState:
    prompt = REASONER_PROMPT_TEMPLATE.format(
        context=state["retrieved_context"] or "No context retrieved.",
        concepts=", ".join(state["graph_concepts"]) or "None",
        query=state["query"]
    )

    response = ollama.generate(model=OLLAMA_MODEL, prompt=prompt)
    raw_text = response["response"]

    claims = re.findall(r"\[CLAIM:\s*(.*?)\]", raw_text)

    print(f"[Reasoner] Generated {len(claims)} tagged claims")

    state["raw_recommendation"] = raw_text
    state["claims"] = [{"text": c, "status": "Unverifiable"} for c in claims]
    return state

## 12. Symbolic Verifier Agent

Checks each claim's text against the Neo4j graph — a simplified keyword-overlap check against known concept names (a fuller version would re-run NER and linking per claim; this keeps the first pass fast).

In [15]:
def verifier_agent(state: AgentState) -> AgentState:
    known_concepts = [c.lower() for c in state["graph_concepts"]]
    verified = []

    for claim in state["claims"]:
        claim_lower = claim["text"].lower()
        if any(concept in claim_lower for concept in known_concepts):
            claim["status"] = "Verified"
        elif state["retrieved_context"] and any(
            word in state["retrieved_context"].lower() for word in claim_lower.split() if len(word) > 5
        ):
            claim["status"] = "Partially Verified"
        else:
            claim["status"] = "Unverifiable"
        verified.append(claim)

    print(f"[Verifier] Checked {len(verified)} claims against the knowledge graph")
    state["verified_claims"] = verified
    return state

## 13. Autonomous Clinical Auditor Agent

Produces the final plain-English report and computes the ATCS score.

In [16]:
def auditor_agent(state: AgentState) -> AgentState:
    claims = state["verified_claims"]
    total = len(claims)
    verified_count = sum(1 for c in claims if c["status"] == "Verified")
    atcs = round((verified_count / total) * 100, 1) if total > 0 else 0.0

    lines = [
        "=== CLINICAL AUDIT REPORT ===",
        f"Query: {state['query']}",
        "",
        "Recommendation:",
        state["raw_recommendation"],
        "",
        "Claim-by-claim verification:"
    ]
    for c in claims:
        lines.append(f"  [{c['status']}] {c['text']}")

    lines.append("")
    lines.append(f"Audit Trail Coverage Score (ATCS): {atcs}%  ({verified_count}/{total} claims verified)")

    report = "\n".join(lines)
    print("[Auditor] Report generated")

    state["audit_report"] = report
    return state

## 14. Wire up the LangGraph pipeline

In [17]:
graph_builder = StateGraph(AgentState)

graph_builder.add_node("orchestrator", orchestrator_agent)
graph_builder.add_node("retriever", retriever_agent)
graph_builder.add_node("reasoner", reasoning_agent)
graph_builder.add_node("verifier", verifier_agent)
graph_builder.add_node("auditor", auditor_agent)

graph_builder.set_entry_point("orchestrator")
graph_builder.add_edge("orchestrator", "retriever")
graph_builder.add_edge("retriever", "reasoner")
graph_builder.add_edge("reasoner", "verifier")
graph_builder.add_edge("verifier", "auditor")
graph_builder.add_edge("auditor", END)

clinical_pipeline = graph_builder.compile()

print("LangGraph pipeline compiled.")

LangGraph pipeline compiled.


## 15. Run the full pipeline

In [18]:
import re

initial_state: AgentState = {
    "query": "What should be considered for a patient presenting with COVID-19 and respiratory distress?",
    "retrieved_context": "",
    "graph_concepts": [],
    "raw_recommendation": "",
    "claims": [],
    "verified_claims": [],
    "audit_report": ""
}

final_state = clinical_pipeline.invoke(initial_state)

print("\n\n")
print(final_state["audit_report"])

driver.close()

[Orchestrator] Received query: What should be considered for a patient presenting with COVID-19 and respiratory distress?
[Retriever] 5 vector matches, 5 graph matches
[Reasoner] Generated 0 tagged claims
[Verifier] Checked 0 claims against the knowledge graph
[Auditor] Report generated



=== CLINICAL AUDIT REPORT ===
Query: What should be considered for a patient presenting with COVID-19 and respiratory distress?

Recommendation:
When managing a patient presenting with COVID-19 and respiratory distress, the following considerations are essential:

1. **Assess Overall Health Status**: Determine if the patient is in a critical condition. If so, mechanical ventilation may be necessary. If not, non-invasive options like nasal cannulas could suffice.

2. **Evaluate Comorbidities**: Consider the patient's existing health conditions, such as heart failure, diabetes, or asthma, which may influence treatment approach and require more aggressive measures.

3. **Assess Complication Risk**: If t

## Next steps

1. Try 2-3 more queries end-to-end and inspect the audit reports, choosing queries that match concepts actually present in the current Module 2 subset.
2. **Swapping in DeepSeek-r1:32b:** this needs a Colab Pro session with an A100 GPU (40GB VRAM) rather than the free T4 (16GB VRAM is not enough for 32b). On that A100 session, only `reasoning_agent`'s model call changes — nothing else in the graph needs to change. This is worth treating as an occasional, deliberate comparison run rather than the default, given the cost of Colab Pro.
3. Module 5 formalises the Auditor Agent's output into the polished, thesis-ready report format with proper ATCS methodology documentation.
4. Save this notebook into `ClinicalTrust/notebooks/` on Drive alongside Modules 1–3.